# LEONI MED — Industrial Connector Inspection
## Binary Classification: Piece Conforme vs Piece Defect

This notebook trains a transfer-learning model (**EfficientNetB0**) to classify
electrical connector terminal images as **Normal** (`piece conforme`) or
**Defective** (`piece defect`).

**Pipeline overview**
1. Load dataset with `image_dataset_from_directory`
2. Split into training (80%) / validation (20%)
3. Apply data augmentation (training set only)
4. Build EfficientNetB0-based classifier with a frozen backbone
5. Train with EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
6. Evaluate: accuracy/loss curves, confusion matrix, classification report
7. Export `best_model.keras` and `class_names.json`


## 1. Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------------------
DATASET_DIR = os.path.join("LEONI_MED", "piece")   # root folder containing the two class subfolders
IMG_SIZE = (224, 224)                              # EfficientNetB0 native input size
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
SEED = 123
EPOCHS = 30
AUTOTUNE = tf.data.AUTOTUNE

MODEL_OUTPUT_PATH = "best_model.keras"
CLASS_NAMES_PATH = "class_names.json"

assert os.path.isdir(DATASET_DIR), (
    f"Dataset directory not found: '{DATASET_DIR}'. "
    "Make sure the notebook is run from the folder containing 'LEONI_MED/piece/'."
)


## 3. Load the Dataset

The dataset is loaded directly from the folder structure using
`image_dataset_from_directory`, which automatically infers the two classes
(`piece conforme`, `piece defect`) from the subfolder names.


In [ ]:
# ---------------------------------------------------------------------------
# Training split (80%)
# ---------------------------------------------------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    shuffle=True,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

# ---------------------------------------------------------------------------
# Validation split (20%)
# ---------------------------------------------------------------------------
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    shuffle=False,          # keep a fixed, reproducible order for evaluation
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

class_names = train_ds.class_names
print("Detected classes:", class_names)


In [ ]:
# ---------------------------------------------------------------------------
# Persist the class-index mapping for later use (e.g. by the Streamlit app)
# ---------------------------------------------------------------------------
class_indices = {str(i): name for i, name in enumerate(class_names)}

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_indices, f, indent=4)

print(f"Saved class mapping to '{CLASS_NAMES_PATH}':")
print(json.dumps(class_indices, indent=4))


### Sample images from the training set

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = int(labels[i].numpy()[0])
        plt.title(class_names[label_idx])
        plt.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()


## 4. Data Augmentation

Augmentation is applied **only to the training set**, using:
- `RandomFlip`
- `RandomRotation`
- `RandomZoom`
- `RandomContrast`


In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ],
    name="data_augmentation",
)


In [ ]:
# ---------------------------------------------------------------------------
# Build the final input pipelines
# ---------------------------------------------------------------------------

def prepare(ds, training=False):
    # Apply augmentation only during training
    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    # EfficientNet-specific preprocessing (expects 0-255 float input)
    ds = ds.map(
        lambda x, y: (preprocess_input(x), y),
        num_parallel_calls=AUTOTUNE,
    )
    ds = ds.cache()
    if training:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds


train_ds_prepared = prepare(train_ds, training=True)
val_ds_prepared = prepare(val_ds, training=False)


## 5. Model Architecture

**EfficientNetB0** pretrained on ImageNet is used as a frozen feature
extractor, followed by a custom classification head:

`GlobalAveragePooling2D -> Dropout -> Dense(128, relu) -> Dense(1, sigmoid)`


In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False  # Freeze the pretrained backbone

model = models.Sequential(
    [
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="leoni_med_classifier",
)

model.summary()


## 6. Compilation

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=BinaryCrossentropy(),
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall"),
    ],
)


## 7. Callbacks

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1,
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_OUTPUT_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1,
)

callbacks = [early_stopping, reduce_lr, checkpoint]


## 8. Training

In [ ]:
history = model.fit(
    train_ds_prepared,
    validation_data=val_ds_prepared,
    epochs=EPOCHS,
    callbacks=callbacks,
)


## 9. Training Curves

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend(loc="lower right")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend(loc="upper right")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()


## 10. Evaluation: Confusion Matrix & Classification Report

Since the best weights were restored by `EarlyStopping` (and the best
checkpoint was also saved to disk), we evaluate directly on `val_ds_prepared`.


In [ ]:
# ---------------------------------------------------------------------------
# Collect ground-truth labels and predictions on the validation set
# ---------------------------------------------------------------------------
y_true = np.concatenate([y.numpy() for _, y in val_ds_prepared], axis=0).astype(int).ravel()
y_pred_prob = model.predict(val_ds_prepared).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


## 11. Final Save

The best model (highest `val_accuracy`) was already saved automatically by
`ModelCheckpoint` to `best_model.keras`. The cell below re-confirms the save
and verifies the artifacts on disk.


In [ ]:
# Safety net: ensure the final in-memory model is also persisted
model.save(MODEL_OUTPUT_PATH)

print("Artifacts saved:")
print(f"  - Model:        {MODEL_OUTPUT_PATH}  (exists: {os.path.exists(MODEL_OUTPUT_PATH)})")
print(f"  - Class names:  {CLASS_NAMES_PATH}  (exists: {os.path.exists(CLASS_NAMES_PATH)})")
